# lets generate teh markovian analysis adding a boostrap to check if the obtained results are at all significant

### we first import the data and set some important conditions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df_analysis = pd.read_csv("../../data/markovian_analysis_baseline.csv")

state_order = ["comfortable-neutral", "uncomfortable", "very uncomfortable"]
MAX_STOP = 5

In [ ]:
"""
LSTM per predir comfort3_option1 — executa localment amb PyTorch
Requisits: pip install torch pandas numpy scikit-learn
"""

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import GroupShuffleSplit
import warnings
warnings.filterwarnings('ignore')

# ── 1. CÀRREGA I PREPROCESSAMENT ────────────────────────────────────────────

df = pd.read_csv('../../data/markovian_analysis_baseline.csv')
df = df.sort_values(['ID', 'stop_idx']).reset_index(drop=True)

thermal_sensation_order = [
    'Very cold','Cold','Slightly cool','Cool','Neutral',
    'Slightly warm','Warm','Hot','Very hot'
]
thermal_comfort_walk_order = [
    'Very uncomfortable','Uncomfortable','Slightly uncomfortable',
    'Neutral','Slightly comfortable','Comfortable','Very comfortable'
]

def encode_ordinal(series, order):
    return series.map({v: i for i, v in enumerate(order)}).astype(float)

df['thermal_sensation_enc']       = encode_ordinal(df['thermal_sensation'], thermal_sensation_order)
df['thermal_comfort_walking_enc'] = encode_ordinal(df['thermal_comfort_walking'], thermal_comfort_walk_order)

# Stop 1 no té thermal_comfort_walking → imputa amb 3.0 (Neutral)
df['thermal_comfort_walking_enc'] = df['thermal_comfort_walking_enc'].fillna(3.0)

# Features estàtiques per persona (invariants al llarg de la seqüència)
STATIC_FEATURES = ['gender', 'age', 'participants_clothing']

# Features temporals per timestep
TEMPORAL_FEATURES = [
    'thermal_sensation_enc',
    'thermal_comfort_walking_enc',
    'sun', 'wind', 'SVF_1.5m', 'NDVI_1.5m',
    'temp_rel_walk', 'HDX_rel_walk',
    'climate_shelter', 'surface_type', 'space_category',
]

TARGET = 'comfort3_option1'

# Encode categoricals
le_gender   = LabelEncoder()
le_clothing = LabelEncoder()
le_shelter  = LabelEncoder()
le_surface  = LabelEncoder()
le_space    = LabelEncoder()
le_y        = LabelEncoder()

df['gender']               = le_gender.fit_transform(df['gender'].astype(str))
df['participants_clothing']= le_clothing.fit_transform(df['participants_clothing'].astype(str))
df['climate_shelter']      = le_shelter.fit_transform(df['climate_shelter'].astype(str))
df['surface_type']         = le_surface.fit_transform(df['surface_type'].astype(str))
df['space_category']       = le_space.fit_transform(df['space_category'].astype(str))
df['y_enc']                = le_y.fit_transform(df[TARGET])

# age és un rang categòric ("13-15", "Less than 10"...) → convertim al punt mig
def age_to_midpoint(val):
    val = str(val).strip()
    if val == 'Less than 10':
        return 8.0
    if '-' in val:
        parts = val.split('-')
        try:
            return (float(parts[0]) + float(parts[1])) / 2
        except ValueError:
            return np.nan
    try:
        return float(val)
    except ValueError:
        return np.nan

df['age'] = df['age'].apply(age_to_midpoint)

# sun i wind són categòrics ordinals → encode numèric amb ordre lògic
sun_order  = ['In full shade', 'In a mixture of sun and shadow', 'In full sun']
wind_order = ["It's not windy", 'A very light wind', 'A light wind', 'A moderate wind', 'A strong wind']
df['sun']  = encode_ordinal(df['sun'],  sun_order).fillna(1.0)
df['wind'] = encode_ordinal(df['wind'], wind_order).fillna(1.0)

# Normalitza totes les features numèriques contínues
cont_cols = ['age', 'sun', 'wind', 'SVF_1.5m', 'NDVI_1.5m', 'temp_rel_walk', 'HDX_rel_walk']
for col in cont_cols:
    mu, sigma = df[col].mean(), df[col].std() + 1e-8
    df[col] = (df[col] - mu) / sigma
# NaN residuals (e.g. temp_rel_walk, HDX_rel_walk) → imputa amb 0 (= mitjana)
df[cont_cols] = df[cont_cols].fillna(0.0)

# Classe: índex de 'very uncomfortable' per als pesos
vu_idx = list(le_y.classes_).index('very uncomfortable')
u_idx  = list(le_y.classes_).index('uncomfortable')
cn_idx = list(le_y.classes_).index('comfortable-neutral')
print('Classes:', le_y.classes_)
print(f'  cn={cn_idx}, u={u_idx}, vu={vu_idx}')

# ── 2. DATASET PER PERSONA ───────────────────────────────────────────────────

MAX_LEN = int(df.groupby('ID')['stop_idx'].count().max())  # 7
N_TEMPORAL = len(TEMPORAL_FEATURES)
N_STATIC   = len(STATIC_FEATURES)

class ComfortDataset(Dataset):
    """
    Cada element és una persona (ID).
    Retorna:
      temporal  [MAX_LEN, N_TEMPORAL]  — seqüència de features per stop
      static    [N_STATIC]             — features invariants
      labels    [MAX_LEN]              — target per stop (pad amb -1)
      mask      [MAX_LEN]              — 1 si el timestep és real, 0 si és padding
    """
    def __init__(self, ids, df):
        self.ids = ids
        self.df  = df

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        pid   = self.ids[idx]
        rows  = self.df[self.df['ID'] == pid].sort_values('stop_idx')
        T     = len(rows)

        temporal = np.zeros((MAX_LEN, N_TEMPORAL), dtype=np.float32)
        labels   = np.full(MAX_LEN, -1, dtype=np.int64)
        mask     = np.zeros(MAX_LEN, dtype=np.float32)

        for t, (_, row) in enumerate(rows.iterrows()):
            temporal[t] = [row[f] for f in TEMPORAL_FEATURES]
            labels[t]   = int(row['y_enc'])
            mask[t]     = 1.0

        static = np.array([rows.iloc[0][f] for f in STATIC_FEATURES], dtype=np.float32)

        return (
            torch.tensor(temporal),
            torch.tensor(static),
            torch.tensor(labels),
            torch.tensor(mask),
        )


# ── 3. MODEL LSTM ────────────────────────────────────────────────────────────

class ComfortLSTM(nn.Module):
    def __init__(self, n_temporal, n_static, hidden_size=64, num_layers=2,
                 dropout=0.3, n_classes=3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_temporal,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.static_proj = nn.Sequential(
            nn.Linear(n_static, 16),
            nn.ReLU(),
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size + 16, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, n_classes),
        )

    def forward(self, temporal, static):
        # temporal: [B, T, n_temporal]
        lstm_out, _ = self.lstm(temporal)          # [B, T, hidden]
        static_feat = self.static_proj(static)     # [B, 16]
        static_exp  = static_feat.unsqueeze(1).expand(-1, lstm_out.size(1), -1)  # [B, T, 16]
        combined    = torch.cat([lstm_out, static_exp], dim=-1)  # [B, T, hidden+16]
        logits      = self.classifier(combined)    # [B, T, n_classes]
        return logits


# ── 4. ENTRENAMENT ───────────────────────────────────────────────────────────

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for temporal, static, labels, mask in loader:
        temporal = temporal.to(device)
        static   = static.to(device)
        labels   = labels.to(device)
        mask     = mask.to(device)

        optimizer.zero_grad()
        logits = model(temporal, static)           # [B, T, C]
        B, T, C = logits.shape

        # Flatten, ignorar timesteps de padding (label == -1)
        logits_flat = logits.view(B * T, C)
        labels_flat = labels.view(B * T)
        mask_flat   = mask.view(B * T).bool()

        logits_valid = logits_flat[mask_flat]
        labels_valid = labels_flat[mask_flat]

        loss = criterion(logits_valid, labels_valid)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, device, le_y):
    model.eval()
    all_true, all_pred = [], []
    for temporal, static, labels, mask in loader:
        temporal = temporal.to(device)
        static   = static.to(device)
        labels   = labels.to(device)
        mask     = mask.to(device)

        logits = model(temporal, static)
        preds  = logits.argmax(dim=-1)             # [B, T]

        B, T = preds.shape
        mask_flat  = mask.view(B * T).bool().cpu()
        preds_flat = preds.view(B * T).cpu()
        labels_flat= labels.view(B * T).cpu()

        all_true.extend(labels_flat[mask_flat].numpy())
        all_pred.extend(preds_flat[mask_flat].numpy())

    y_true_labels = le_y.inverse_transform(all_true)
    y_pred_labels = le_y.inverse_transform(all_pred)
    return y_true_labels, y_pred_labels


# ── 5. PIPELINE COMPLET ──────────────────────────────────────────────────────

def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Device: {device}')

    all_ids = df['ID'].unique()

    # Pesos de classe per combatre desbalanceig (w_vu=8 optim del grid search)
    class_weights = torch.ones(3, dtype=torch.float32)
    class_weights[vu_idx] = 8.0
    class_weights[u_idx]  = 1.5
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

    # GroupShuffleSplit per persona (20% test, persones no repetides)
    # — equivalent al LOGO però manejable en temps
    gss = GroupShuffleSplit(n_splits=10, test_size=0.2, random_state=42)
    # 'groups' ha de ser per ID (una etiqueta per persona)
    person_df = df.drop_duplicates('ID')[['ID']].copy()
    person_df['group'] = person_df['ID']

    fold_f1_macro, fold_f1_vu = [], []

    for fold, (train_idx, test_idx) in enumerate(
        gss.split(person_df['ID'], groups=person_df['group'])
    ):
        train_ids = person_df.iloc[train_idx]['ID'].values
        test_ids  = person_df.iloc[test_idx]['ID'].values

        train_ds = ComfortDataset(train_ids, df)
        test_ds  = ComfortDataset(test_ids,  df)
        train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
        test_dl  = DataLoader(test_ds,  batch_size=64, shuffle=False)

        model = ComfortLSTM(
            n_temporal=N_TEMPORAL,
            n_static=N_STATIC,
            hidden_size=64,
            num_layers=2,
            dropout=0.3,
            n_classes=3,
        ).to(device)

        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, patience=5, factor=0.5, min_lr=1e-5
        )

        best_f1, best_state = 0, None
        for epoch in range(60):
            loss = train_one_epoch(model, train_dl, optimizer, criterion, device)
            y_true, y_pred = evaluate(model, test_dl, device, le_y)
            f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
            scheduler.step(-f1)
            if f1 > best_f1:
                best_f1 = f1
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
            if (epoch + 1) % 10 == 0:
                f1_vu = f1_score(y_true, y_pred,
                                 labels=['very uncomfortable'], average='macro', zero_division=0)
                print(f'  Fold {fold+1} Epoch {epoch+1:3d} | loss={loss:.4f} '
                      f'F1macro={f1:.4f} F1_vu={f1_vu:.4f}')

        # Avalua amb el millor estat del fold
        model.load_state_dict(best_state)
        y_true, y_pred = evaluate(model, test_dl, device, le_y)
        f1m  = f1_score(y_true, y_pred, average='macro', zero_division=0)
        f1vu = f1_score(y_true, y_pred, labels=['very uncomfortable'],
                        average='macro', zero_division=0)
        fold_f1_macro.append(f1m)
        fold_f1_vu.append(f1vu)
        print(f'Fold {fold+1} FINAL → F1macro={f1m:.4f}  F1_vu={f1vu:.4f}')

    print('\n' + '='*55)
    print('RESULTAT FINAL LSTM (10 folds, persones separades)')
    print(f'  F1 macro:         {np.mean(fold_f1_macro):.4f} ± {np.std(fold_f1_macro):.4f}')
    print(f'  F1 very uncomf:   {np.mean(fold_f1_vu):.4f} ± {np.std(fold_f1_vu):.4f}')
    print('='*55)

    # Report final en el darrer fold (indicatiu)
    print('\nClassification report (últim fold):')
    print(classification_report(y_true, y_pred))


if __name__ == '__main__':
    main()


# 1.0 Building transitions and triplets

In [ ]:
def build_transitions_and_triplets(df, state_col, state_order, max_stop=5):
    state_num_map = {s: i for i, s in enumerate(state_order)}

    df = df.copy()
    df = df.sort_values(["ID", "walk_id", "stop_idx", "row_id"]).reset_index(drop=True)

    df["state"] = df[state_col]
    df["state_num"] = df["state"].map(state_num_map)

    g = df.groupby(["ID", "walk_id"], sort=False)

    df["prev_state"] = g["state"].shift(1)
    df["prev_state_num"] = g["state_num"].shift(1)
    df["prev_stop_idx"] = g["stop_idx"].shift(1)

    df["next_state"] = g["state"].shift(-1)
    df["next_state_num"] = g["state_num"].shift(-1)
    df["next_stop_idx"] = g["stop_idx"].shift(-1)

    transitions = df[
        (df["next_stop_idx"] - df["stop_idx"] == 1) &
        (df["stop_idx"] <= max_stop) &
        (df["next_stop_idx"] <= max_stop)
    ].copy()

    transitions["delta"] = transitions["next_state_num"] - transitions["state_num"]

    triplets = df[
        (df["stop_idx"] <= max_stop) &
        (df["prev_stop_idx"] >= 1) &
        (df["next_stop_idx"] <= max_stop) &
        ((df["stop_idx"] - df["prev_stop_idx"]) == 1) &
        ((df["next_stop_idx"] - df["stop_idx"]) == 1)
    ].copy()

    triplets["delta_prev_to_curr"] = triplets["state_num"] - triplets["prev_state_num"]
    triplets["delta_curr_to_next"] = triplets["next_state_num"] - triplets["state_num"]

    return transitions, triplets

# 2.0 Now we build the stationary distribution from the data distribution 

In [ ]:
def stationary_distribution(P):
    M = P.values.astype(float)
    n = M.shape[0]

    A = M.T - np.eye(n)
    A[-1, :] = 1.0
    b = np.zeros(n)
    b[-1] = 1.0

    pi = np.linalg.solve(A, b)
    pi = np.clip(pi, 0, None)
    pi = pi / pi.sum()

    return pd.Series(pi, index=P.index, name="pi_stat")

# molaria poder mirar si fittejant a second order stat. distribution queda millorr?

# 3.0 We then get the results for the O(1) amd O(2) Markov chains

In [ ]:
def compute_markov_memory_summary(transitions, triplets, state_order):
    cm1_counts = pd.crosstab(
        transitions["state"],
        transitions["next_state"]
    ).reindex(index=state_order, columns=state_order, fill_value=0)

    P1 = cm1_counts.div(cm1_counts.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)

    cm2_counts = (
        triplets.groupby(["prev_state", "state", "next_state"])
        .size()
        .rename("count")
        .reset_index()
    )

    cm2_counts["prob"] = cm2_counts.groupby(["prev_state", "state"])["count"].transform(
        lambda x: x / x.sum()
    )

    rows = []
    rows_long = []

    for prev_state in state_order:
        for state in state_order:
            sub = cm2_counts[
                (cm2_counts["prev_state"] == prev_state) &
                (cm2_counts["state"] == state)
            ].copy()

            if sub.empty:
                continue

            p2 = sub.set_index("next_state")["prob"].reindex(state_order, fill_value=0.0)
            p1 = P1.loc[state].reindex(state_order, fill_value=0.0)

            n = int(sub["count"].sum())
            l1 = float(np.abs(p2 - p1).sum())
            max_abs = float(np.abs(p2 - p1).max())

            rows.append({
                "prev_state": prev_state,
                "state": state,
                "n_triplets": n,
                "L1_diff_order2_vs_order1": l1,
                "max_abs_diff": max_abs,
            })

            for ns in state_order:
                rows_long.append({
                    "prev_state": prev_state,
                    "state": state,
                    "next_state": ns,
                    "n_triplets": n,
                    "L1_diff_order2_vs_order1": l1,
                    "max_abs_diff": max_abs,
                    "p1_prob": float(p1[ns]),
                    "p2_prob": float(p2[ns]),
                    "signed_diff": float(p2[ns] - p1[ns]),
                    "abs_diff": float(abs(p2[ns] - p1[ns])),
                })

    markov_compare = pd.DataFrame(rows).sort_values(
        ["L1_diff_order2_vs_order1", "n_triplets"],
        ascending=[False, False]
    )

    markov_compare_long = pd.DataFrame(rows_long)

    weighted_l1 = np.average(
        markov_compare["L1_diff_order2_vs_order1"],
        weights=markov_compare["n_triplets"]
    )

    by_current = (
        markov_compare.groupby("state")
        .apply(lambda d: np.average(d["L1_diff_order2_vs_order1"], weights=d["n_triplets"]))
        .rename("weighted_L1")
        .reset_index()
    )

    return {
        "P1": P1,
        "cm2_counts": cm2_counts,
        "markov_compare": markov_compare,
        "markov_compare_long": markov_compare_long,
        "weighted_mean_L1": float(weighted_l1),
        "weighted_L1_by_current": by_current,
    }

# 4.0 we finally get the out-of-eq summary of what we got

In [ ]:
def compute_noneq_summary(df, transitions, state_col, state_order):
    eps = 1e-12

    cm_counts = pd.crosstab(
        transitions["state"],
        transitions["next_state"]
    ).reindex(index=state_order, columns=state_order, fill_value=0)

    P = cm_counts.div(
        cm_counts.sum(axis=1).replace(0, np.nan),
        axis=0
    ).fillna(0)

    pi_source = (
        transitions["state"]
        .value_counts(normalize=True)
        .reindex(state_order, fill_value=0)
    )

    pi_occ = (
        df[state_col]
        .value_counts(normalize=True)
        .reindex(state_order, fill_value=0)
    )

    pi_stat = stationary_distribution(P).reindex(state_order)

    landscape = pd.DataFrame({
        "state": state_order,
        "pi_source": pi_source.values,
        "pi_occ": pi_occ.values,
        "pi_stat": pi_stat.values,
    })
    landscape["U_source"] = -np.log(np.clip(landscape["pi_source"], eps, None))
    landscape["U_occ"] = -np.log(np.clip(landscape["pi_occ"], eps, None))
    landscape["U_stat"] = -np.log(np.clip(landscape["pi_stat"], eps, None))

    # model currents
    pi_vec = pi_stat.to_numpy(dtype=float)
    P_mat = P.to_numpy(dtype=float)
    flow = pi_vec[:, None] * P_mat
    J_model = flow - flow.T

    # empirical currents
    N = cm_counts.copy()
    N_total = N.values.sum()
    F_emp = N / N_total
    J_emp = F_emp - F_emp.T

    emp_pairs = []
    for i, si in enumerate(state_order):
        for j, sj in enumerate(state_order):
            if i < j:
                fij = float(F_emp.iloc[i, j])
                fji = float(F_emp.iloc[j, i])
                Jij = float(J_emp.iloc[i, j])
                emp_pairs.append({
                    "state_i": si,
                    "state_j": sj,
                    "count_ij": int(N.iloc[i, j]),
                    "count_ji": int(N.iloc[j, i]),
                    "F_ij_emp": fij,
                    "F_ji_emp": fji,
                    "J_ij_emp": Jij,
                    "abs_J_emp": abs(Jij),
                    "rel_J_emp": abs(Jij) / max(fij + fji, eps),
                    "direction": f"{si}->{sj}" if Jij > 0 else f"{sj}->{si}",
                })

    emp_currents = pd.DataFrame(emp_pairs).sort_values("abs_J_emp", ascending=False)

    resilience = pd.Series({
    "onset_of_discomfort": (
        P.loc["comfortable-neutral", "uncomfortable"] +
        P.loc["comfortable-neutral", "very uncomfortable"]
    ),
    "escalation_to_very_uncomfortable": P.loc["uncomfortable", "very uncomfortable"],
    "trap_persistence_very_uncomfortable": P.loc["very uncomfortable", "very uncomfortable"],
    "recovery_from_uncomfortable": P.loc["uncomfortable", "comfortable-neutral"],
    "escape_from_very_uncomfortable": (
        P.loc["very uncomfortable", "comfortable-neutral"] +
        P.loc["very uncomfortable", "uncomfortable"]
            ),
        }, name="value")

    return {
        "P": P,
        "pi_source": pi_source,
        "pi_occ": pi_occ,
        "pi_stat": pi_stat,
        "landscape": landscape,
        "emp_currents": emp_currents,
        "resilience": resilience,
    }

# 5.0 Definim la funció que ens treu tots els valors

In [ ]:
def analyze_markov_subset(df, state_col, state_order, max_stop=5):
    transitions, triplets = build_transitions_and_triplets(
        df=df,
        state_col=state_col,
        state_order=state_order,
        max_stop=max_stop
    )

    memory = compute_markov_memory_summary(
        transitions=transitions,
        triplets=triplets,
        state_order=state_order
    )

    noneq = compute_noneq_summary(
        df=df[df["stop_idx"] <= max_stop].copy(),
        transitions=transitions,
        state_col=state_col,
        state_order=state_order
    )

    return {
        "transitions": transitions,
        "triplets": triplets,
        **memory,
        **noneq,
    }

# i la cridem

In [ ]:
results_3 = analyze_markov_subset(
    df=df_analysis,
    state_col="comfort3_option1",
    state_order=state_order,
    max_stop=5
)

print(results_3["transitions"].shape)
print(results_3["triplets"].shape)
print(results_3["landscape"])
print(results_3["markov_compare"])
print(results_3["weighted_mean_L1"])
print(results_3["weighted_L1_by_current"])
print(results_3["emp_currents"])
print(results_3["resilience"])

# 2na Secció: Boostrap / Permutations

## Boostrap

In [ ]:
def bootstrap_markov_memory(df_analysis, state_col, state_order, n_boot=500, seed=123):
    rng = np.random.default_rng(seed)

    df = df_analysis.copy()
    df["traj_id"] = df["ID"].astype(str) + "||" + df["walk_id"].astype(str)
    traj_ids = df["traj_id"].drop_duplicates().tolist()

    rows_global = []
    rows_current = []

    for b in range(n_boot):
        sampled_ids = rng.choice(traj_ids, size=len(traj_ids), replace=True)

        parts = []
        for k, tid in enumerate(sampled_ids):
            sub = df[df["traj_id"] == tid].copy()
            new_id = f"{tid}__boot{k}"
            sub["ID"] = new_id
            sub["walk_id"] = new_id
            parts.append(sub)

        df_boot = pd.concat(parts, ignore_index=True)

        res = analyze_markov_subset(
            df=df_boot,
            state_col=state_col,
            state_order=state_order,
            max_stop=5
        )

        rows_global.append({
            "boot": b,
            "weighted_mean_L1": res["weighted_mean_L1"]
        })

        tmp = res["weighted_L1_by_current"].copy()
        tmp["boot"] = b
        rows_current.append(tmp)

    boot_global = pd.DataFrame(rows_global)
    boot_current = pd.concat(rows_current, ignore_index=True)

    return boot_global, boot_current

In [ ]:
#boot_global_3, boot_current_3 = bootstrap_markov_memory(
 #   df_analysis=df_analysis,
  #  state_col="comfort3_option1",
   # state_order=state_order,
   # n_boot=500,
   # seed=123
#)

In [ ]:
#boot_global_3["weighted_mean_L1"].quantile([0.025, 0.5, 0.975])

In [ ]:
#boot_current_3.groupby("state")["weighted_L1"].quantile([0.025, 0.5, 0.975]).unstack()

## Ara probem premutant estats previs (null hypotesis for memory)

In [ ]:
def permutation_markov_memory(df_analysis, state_col, state_order, n_perm=1000, seed=123):
    rng = np.random.default_rng(seed)

    transitions, triplets = build_transitions_and_triplets(
        df_analysis, state_col=state_col, state_order=state_order, max_stop=5
    )

    observed = compute_markov_memory_summary(
        transitions, triplets, state_order
    )["weighted_mean_L1"]

    cm1_counts = pd.crosstab(
        transitions["state"], transitions["next_state"]
    ).reindex(index=state_order, columns=state_order, fill_value=0)

    P1 = cm1_counts.div(cm1_counts.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)

    perm_vals = []

    for b in range(n_perm):
        trip_perm = triplets.copy()
        trip_perm["prev_state_perm"] = (
            trip_perm.groupby("state")["prev_state"]
            .transform(lambda x: rng.permutation(x.values))
        )

        cm2_counts = (
            trip_perm.groupby(["prev_state_perm", "state", "next_state"])
            .size()
            .rename("count")
            .reset_index()
            .rename(columns={"prev_state_perm": "prev_state"})
        )
        cm2_counts["prob"] = cm2_counts.groupby(["prev_state", "state"])["count"].transform(
            lambda x: x / x.sum()
        )

        rows = []
        for prev_state in state_order:
            for state in state_order:
                sub = cm2_counts[
                    (cm2_counts["prev_state"] == prev_state) &
                    (cm2_counts["state"] == state)
                ].copy()

                if sub.empty:
                    continue

                p2 = sub.set_index("next_state")["prob"].reindex(state_order, fill_value=0.0)
                p1 = P1.loc[state].reindex(state_order, fill_value=0.0)

                n = int(sub["count"].sum())
                l1 = float(np.abs(p2 - p1).sum())

                rows.append({
                    "prev_state": prev_state,
                    "state": state,
                    "n_triplets": n,
                    "L1_diff_order2_vs_order1": l1
                })

        markov_compare_perm = pd.DataFrame(rows)
        weighted_l1_perm = np.average(
            markov_compare_perm["L1_diff_order2_vs_order1"],
            weights=markov_compare_perm["n_triplets"]
        )
        perm_vals.append(weighted_l1_perm)

    perm_vals = np.array(perm_vals)
    p_value = np.mean(perm_vals >= observed)

    return observed, perm_vals, p_value

In [ ]:
observed_l1, perm_vals, p_value = permutation_markov_memory(
    df_analysis=df_analysis,
    state_col="comfort3_option1",
    state_order=state_order,
    n_perm=400,
    seed=123
)

print("Observed weighted L1:", observed_l1)
print("Permutation p-value:", p_value)

## sembla quela memòria (en general) és aquí i no és un fruit estadístic.

# Mirem-ho ara per estat

In [ ]:
import numpy as np
import pandas as pd

def permutation_markov_memory_by_current_state(
    transitions,
    triplets,
    state_order,
    n_perm=5000,
    seed=123
):
    rng = np.random.default_rng(seed)

    # ---------- O(1): P(next | current) ----------
    cm1_counts = pd.crosstab(
        transitions["state"],
        transitions["next_state"]
    ).reindex(index=state_order, columns=state_order, fill_value=0)

    P1 = cm1_counts.div(
        cm1_counts.sum(axis=1).replace(0, np.nan),
        axis=0
    ).fillna(0)

    # ---------- Observed state-specific weighted L1 ----------
    observed_rows = []

    cm2_counts_obs = (
        triplets.groupby(["prev_state", "state", "next_state"])
        .size()
        .rename("count")
        .reset_index()
    )
    cm2_counts_obs["prob"] = cm2_counts_obs.groupby(["prev_state", "state"])["count"].transform(
        lambda x: x / x.sum()
    )

    for current_state in state_order:
        rows_state = []

        for prev_state in state_order:
            sub = cm2_counts_obs[
                (cm2_counts_obs["prev_state"] == prev_state) &
                (cm2_counts_obs["state"] == current_state)
            ].copy()

            if sub.empty:
                continue

            p2 = (
                sub.set_index("next_state")["prob"]
                .reindex(state_order, fill_value=0.0)
            )
            p1 = P1.loc[current_state].reindex(state_order, fill_value=0.0)

            n = int(sub["count"].sum())
            l1 = float(np.abs(p2 - p1).sum())

            rows_state.append({
                "prev_state": prev_state,
                "state": current_state,
                "n_triplets": n,
                "L1": l1
            })

        if len(rows_state) == 0:
            continue

        df_state = pd.DataFrame(rows_state)
        observed_weighted_l1 = np.average(df_state["L1"], weights=df_state["n_triplets"])

        observed_rows.append({
            "state": current_state,
            "observed_weighted_L1": float(observed_weighted_l1),
            "total_triplets": int(df_state["n_triplets"].sum())
        })

    observed_df = pd.DataFrame(observed_rows)

    # ---------- Permutation ----------
    perm_rows = []

    for b in range(n_perm):
        trip_perm = triplets.copy()

        # shuffle prev_state within each current state
        trip_perm["prev_state_perm"] = (
            trip_perm.groupby("state")["prev_state"]
            .transform(lambda x: rng.permutation(x.values))
        )

        cm2_counts_perm = (
            trip_perm.groupby(["prev_state_perm", "state", "next_state"])
            .size()
            .rename("count")
            .reset_index()
            .rename(columns={"prev_state_perm": "prev_state"})
        )

        cm2_counts_perm["prob"] = cm2_counts_perm.groupby(["prev_state", "state"])["count"].transform(
            lambda x: x / x.sum()
        )

        for current_state in state_order:
            rows_state = []

            for prev_state in state_order:
                sub = cm2_counts_perm[
                    (cm2_counts_perm["prev_state"] == prev_state) &
                    (cm2_counts_perm["state"] == current_state)
                ].copy()

                if sub.empty:
                    continue

                p2 = (
                    sub.set_index("next_state")["prob"]
                    .reindex(state_order, fill_value=0.0)
                )
                p1 = P1.loc[current_state].reindex(state_order, fill_value=0.0)

                n = int(sub["count"].sum())
                l1 = float(np.abs(p2 - p1).sum())

                rows_state.append({
                    "prev_state": prev_state,
                    "state": current_state,
                    "n_triplets": n,
                    "L1": l1
                })

            if len(rows_state) == 0:
                continue

            df_state = pd.DataFrame(rows_state)
            weighted_l1_perm = np.average(df_state["L1"], weights=df_state["n_triplets"])

            perm_rows.append({
                "perm": b,
                "state": current_state,
                "weighted_L1_perm": float(weighted_l1_perm)
            })

    perm_df = pd.DataFrame(perm_rows)

    # ---------- Compare observed vs null ----------
    summary_rows = []

    for current_state in state_order:
        obs_sub = observed_df[observed_df["state"] == current_state]
        perm_sub = perm_df[perm_df["state"] == current_state]

        if len(obs_sub) == 0 or len(perm_sub) == 0:
            continue

        observed_val = float(obs_sub["observed_weighted_L1"].iloc[0])
        null_vals = perm_sub["weighted_L1_perm"].to_numpy()

        p_value = np.mean(null_vals >= observed_val)

        summary_rows.append({
            "state": current_state,
            "observed_weighted_L1": observed_val,
            "p_value_one_sided": float(p_value),
            "null_q025": float(np.quantile(null_vals, 0.025)),
            "null_median": float(np.quantile(null_vals, 0.5)),
            "null_q975": float(np.quantile(null_vals, 0.975)),
            "total_triplets": int(obs_sub["total_triplets"].iloc[0]),
        })

    summary_df = pd.DataFrame(summary_rows)

    return observed_df, perm_df, summary_df

In [ ]:
transitions, triplets = build_transitions_and_triplets(
        df_analysis, state_col="comfort3_option1", state_order=state_order, max_stop=5
    )

obs_state_mem, perm_state_mem, summary_state_mem = permutation_markov_memory_by_current_state(
    transitions=transitions,
    triplets=triplets,
    state_order=state_order,
    n_perm=1000,
    seed=123
)

summary_state_mem

# Boostrap/Permutacions de les corrents 

## Boostrap: 1r treiem les trajectories senceres per a poder resamplejar correctament

In [ ]:
import numpy as np
import pandas as pd

def extract_state_sequences(df, state_col, max_stop=5):
    df = df.copy()
    df = df[df["stop_idx"] <= max_stop].copy()
    df = df.sort_values(["ID", "walk_id", "stop_idx", "row_id"]).reset_index(drop=True)

    sequences = []

    for (_, _), sub in df.groupby(["ID", "walk_id"], sort=False):
        sub = sub.dropna(subset=[state_col]).copy()
        if len(sub) < 2:
            continue

        stops = sub["stop_idx"].to_numpy()
        states = sub[state_col].tolist()

        current_run = [states[0]]

        for k in range(1, len(states)):
            if stops[k] - stops[k-1] == 1:
                current_run.append(states[k])
            else:
                if len(current_run) >= 2:
                    sequences.append(current_run)
                current_run = [states[k]]

        if len(current_run) >= 2:
            sequences.append(current_run)

    return sequences

### construim la matriu de cuentas a partir de les noves transicions

In [ ]:
def transition_counts_from_sequences(sequences, state_order):
    state_to_idx = {s: i for i, s in enumerate(state_order)}
    N = np.zeros((len(state_order), len(state_order)), dtype=int)

    for seq in sequences:
        for a, b in zip(seq[:-1], seq[1:]):
            if a in state_to_idx and b in state_to_idx:
                i = state_to_idx[a]
                j = state_to_idx[b]
                N[i, j] += 1

    return pd.DataFrame(N, index=state_order, columns=state_order)

### corrents empíriques

In [ ]:
def empirical_currents_from_counts(N, state_order):
    eps = 1e-12

    N_total = N.values.sum()
    F_emp = N / N_total
    J_emp = F_emp - F_emp.T

    rows = []
    for i, si in enumerate(state_order):
        for j, sj in enumerate(state_order):
            if i < j:
                fij = float(F_emp.iloc[i, j])
                fji = float(F_emp.iloc[j, i])
                Jij = float(J_emp.iloc[i, j])

                rows.append({
                    "state_i": si,
                    "state_j": sj,
                    "count_ij": int(N.iloc[i, j]),
                    "count_ji": int(N.iloc[j, i]),
                    "F_ij_emp": fij,
                    "F_ji_emp": fji,
                    "J_ij_emp": Jij,
                    "abs_J_emp": abs(Jij),
                    "rel_J_emp": abs(Jij) / max(fij + fji, eps),
                    "direction": f"{si}->{sj}" if Jij > 0 else f"{sj}->{si}",
                })

    emp_currents = pd.DataFrame(rows).sort_values("abs_J_emp", ascending=False)

    out = {
        "N": N,
        "F_emp": F_emp,
        "J_emp": J_emp,
        "emp_currents": emp_currents,
    }

    # escalar de cicle només per 3 estats
    if len(state_order) == 3:
        j01 = float(J_emp.iloc[0, 1])      # 0->1
        j12 = float(J_emp.iloc[1, 2])      # 1->2
        j20 = float(J_emp.iloc[2, 0])      # 2->0

        cycle_current = (j01 + j12 + j20) / 3.0
        cycle_consistency = np.std([j01, j12, j20])

        out["cycle_summary"] = {
            "j01": j01,
            "j12": j12,
            "j20": j20,
            "cycle_current": cycle_current,
            "cycle_consistency": cycle_consistency,
            "cycle_direction": "0->1->2->0" if cycle_current > 0 else "0->2->1->0"
        }

    return out

### boostrap

In [ ]:
def bootstrap_empirical_currents(df_analysis, state_col, state_order, max_stop=5, n_boot=1000, seed=123):
    rng = np.random.default_rng(seed)

    sequences = extract_state_sequences(df_analysis, state_col=state_col, max_stop=max_stop)
    n_seq = len(sequences)

    pair_rows = []
    cycle_rows = []

    for b in range(n_boot):
        idx = rng.choice(np.arange(n_seq), size=n_seq, replace=True)
        boot_sequences = [sequences[i] for i in idx]

        N_boot = transition_counts_from_sequences(boot_sequences, state_order)
        res_boot = empirical_currents_from_counts(N_boot, state_order)

        emp_curr_boot = res_boot["emp_currents"].copy()
        emp_curr_boot["boot"] = b
        pair_rows.append(emp_curr_boot)

        if "cycle_summary" in res_boot:
            cyc = res_boot["cycle_summary"].copy()
            cyc["boot"] = b
            cycle_rows.append(cyc)

    boot_pairs = pd.concat(pair_rows, ignore_index=True)
    boot_cycle = pd.DataFrame(cycle_rows)

    return boot_pairs, boot_cycle

### funció resum dels resultats

In [ ]:
def summarize_bootstrap_currents(boot_pairs):
    summary = (
        boot_pairs.groupby(["state_i", "state_j"])["J_ij_emp"]
        .quantile([0.025, 0.5, 0.975])
        .unstack()
        .reset_index()
    )
    summary.columns = ["state_i", "state_j", "q025", "median", "q975"]
    return summary

def summarize_bootstrap_cycle(boot_cycle):
    if len(boot_cycle) == 0:
        return None

    summary = boot_cycle["cycle_current"].quantile([0.025, 0.5, 0.975])
    sign_stability = max(
        (boot_cycle["cycle_current"] > 0).mean(),
        (boot_cycle["cycle_current"] < 0).mean()
    )

    return {
        "q025": float(summary.loc[0.025]),
        "median": float(summary.loc[0.5]),
        "q975": float(summary.loc[0.975]),
        "sign_stability": float(sign_stability),
    }

## Observat i executem el boostrap

In [ ]:
state_order = ["comfortable-neutral", "uncomfortable", "very uncomfortable"]

seq_obs = extract_state_sequences(
    df_analysis,
    state_col="comfort3_option1",
    max_stop=5
)

N_obs = transition_counts_from_sequences(seq_obs, state_order)
res_obs = empirical_currents_from_counts(N_obs, state_order)

res_obs["emp_currents"]
res_obs["cycle_summary"]

In [ ]:
boot_pairs, boot_cycle = bootstrap_empirical_currents(
    df_analysis=df_analysis,
    state_col="comfort3_option1",
    state_order=state_order,
    max_stop=5,
    n_boot=100,
    seed=123
)

boot_pair_summary = summarize_bootstrap_currents(boot_pairs)
boot_cycle_summary = summarize_bootstrap_cycle(boot_cycle)

print(boot_pair_summary)
print(boot_cycle_summary)


## Ara fem permutacions

In [ ]:
def permutation_time_reversal_currents(df_analysis, state_col, state_order, max_stop=5, n_perm=5000, seed=123):
    rng = np.random.default_rng(seed)

    sequences = extract_state_sequences(df_analysis, state_col=state_col, max_stop=max_stop)

    # observat
    N_obs = transition_counts_from_sequences(sequences, state_order)
    res_obs = empirical_currents_from_counts(N_obs, state_order)

    obs_pairs = res_obs["emp_currents"].copy()
    obs_cycle = res_obs.get("cycle_summary", None)

    pair_perm_rows = []
    cycle_perm_rows = []

    for b in range(n_perm):
        perm_sequences = []
        for seq in sequences:
            if rng.random() < 0.5:
                perm_sequences.append(seq[::-1])  # revertida
            else:
                perm_sequences.append(seq)        # original

        N_perm = transition_counts_from_sequences(perm_sequences, state_order)
        res_perm = empirical_currents_from_counts(N_perm, state_order)

        tmp = res_perm["emp_currents"].copy()
        tmp["perm"] = b
        pair_perm_rows.append(tmp)

        if "cycle_summary" in res_perm:
            cyc = res_perm["cycle_summary"].copy()
            cyc["perm"] = b
            cycle_perm_rows.append(cyc)

    perm_pairs = pd.concat(pair_perm_rows, ignore_index=True)
    perm_cycle = pd.DataFrame(cycle_perm_rows)

    # p-values de les corrents per parella
    pair_tests = []
    for _, row in obs_pairs.iterrows():
        si = row["state_i"]
        sj = row["state_j"]
        obs_j = row["J_ij_emp"]

        null_vals = perm_pairs[
            (perm_pairs["state_i"] == si) &
            (perm_pairs["state_j"] == sj)
        ]["J_ij_emp"].to_numpy()

        p_two_sided = np.mean(np.abs(null_vals) >= abs(obs_j))

        pair_tests.append({
            "state_i": si,
            "state_j": sj,
            "observed_J": obs_j,
            "p_value_two_sided": float(p_two_sided),
            "null_q025": float(np.quantile(null_vals, 0.025)),
            "null_median": float(np.quantile(null_vals, 0.5)),
            "null_q975": float(np.quantile(null_vals, 0.975)),
        })

    pair_tests = pd.DataFrame(pair_tests)

    # p-value del cicle
    cycle_test = None
    if obs_cycle is not None and len(perm_cycle) > 0:
        obs_c = obs_cycle["cycle_current"]
        null_c = perm_cycle["cycle_current"].to_numpy()
        p_cycle = np.mean(np.abs(null_c) >= abs(obs_c))

        cycle_test = {
            "observed_cycle_current": float(obs_c),
            "p_value_two_sided": float(p_cycle),
            "null_q025": float(np.quantile(null_c, 0.025)),
            "null_median": float(np.quantile(null_c, 0.5)),
            "null_q975": float(np.quantile(null_c, 0.975)),
        }

    return {
        "observed_pairs": obs_pairs,
        "observed_cycle": obs_cycle,
        "perm_pairs": perm_pairs,
        "perm_cycle": perm_cycle,
        "pair_tests": pair_tests,
        "cycle_test": cycle_test,
    }

In [ ]:
perm_current_results = permutation_time_reversal_currents(
    df_analysis=df_analysis,
    state_col="comfort3_option1",
    state_order=state_order,
    max_stop=5,
    n_perm=10000,
    seed=123
)

print(perm_current_results["pair_tests"])
print(perm_current_results["cycle_test"])

# Ara seguirem el mateix patró per veure els forçaments en el sistema

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

state_order = ["comfortable-neutral", "uncomfortable", "very uncomfortable"]
MAX_STOP = 5
SUN_COL = "sun"
STATE_COL = "comfort3_option1"

In [ ]:
def recode_sun_binary(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()

    if x == "In full shade":
        return "not_sun_exposed"
    elif x in ["In a mixture of sun and shadow", "In full sun"]:
        return "sun_exposed"
    else:
        return np.nan

# Construim les transicions amb sol

In [ ]:
def build_transitions_with_sunpair_binary(
    df,
    state_col="comfort3_option1",
    sun_col="sun",
    max_stop=5
):
    df = df.copy()
    df = df.sort_values(["ID", "walk_id", "stop_idx", "row_id"]).reset_index(drop=True)

    df["state"] = df[state_col]

    g = df.groupby(["ID", "walk_id"], sort=False)

    df["next_state"] = g["state"].shift(-1)
    df["next_stop_idx"] = g["stop_idx"].shift(-1)

    df["sun_t"] = df[sun_col].map(recode_sun_binary)
    df["sun_t1"] = g[sun_col].shift(-1).map(recode_sun_binary)

    transitions = df[
        (df["next_stop_idx"] - df["stop_idx"] == 1) &
        (df["stop_idx"] <= max_stop) &
        (df["next_stop_idx"] <= max_stop)
    ].copy()

    transitions = transitions.dropna(subset=["state", "next_state", "sun_t", "sun_t1"]).copy()

    transitions["sun_pair_bin"] = (
        transitions["sun_t"].astype(str) + " -> " + transitions["sun_t1"].astype(str)
    )

    return transitions

# i generem la matriu de transicions

In [ ]:
def transition_matrix_from_subset(trans_sub, state_order):
    N = pd.crosstab(
        trans_sub["state"],
        trans_sub["next_state"]
    ).reindex(index=state_order, columns=state_order, fill_value=0)

    P = N.div(N.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
    return N, P

## i les subsequents mètriques

In [ ]:
def compute_deterioration_metrics(P):
    return pd.Series({
        "onset_of_discomfort": (
            P.loc["comfortable-neutral", "uncomfortable"] +
            P.loc["comfortable-neutral", "very uncomfortable"]
        ),
        "escalation_to_very_uncomfortable": P.loc["uncomfortable", "very uncomfortable"],
        "trap_persistence_very_uncomfortable": P.loc["very uncomfortable", "very uncomfortable"],
        "recovery_from_uncomfortable": P.loc["uncomfortable", "comfortable-neutral"] + P.loc["very uncomfortable", "comfortable-neutral"],
        "escape_from_very_uncomfortable": (
            P.loc["very uncomfortable", "comfortable-neutral"] +
            P.loc["very uncomfortable", "uncomfortable"]
        ),
    }, name="value")

In [ ]:
def analyze_sunpair_binary_metrics(
    df,
    state_col="comfort3_option1",
    sun_col="sun",
    state_order=state_order,
    max_stop=5
):
    transitions = build_transitions_with_sunpair_binary(
        df=df,
        state_col=state_col,
        sun_col=sun_col,
        max_stop=max_stop
    )

    results = {}
    rows_summary = []

    for pair_value, trans_sub in transitions.groupby("sun_pair_bin"):
        N, P = transition_matrix_from_subset(trans_sub, state_order)
        metrics = compute_deterioration_metrics(P)

        state_counts = trans_sub["state"].value_counts().reindex(state_order, fill_value=0)

        results[pair_value] = {
            "transitions": trans_sub,
            "N": N,
            "P": P,
            "metrics": metrics,
            "state_counts": state_counts,
        }

        row = {
            "sun_pair_bin": pair_value,
            "n_transitions": len(trans_sub),
            "n_state_0": int(state_counts["comfortable-neutral"]),
            "n_state_1": int(state_counts["uncomfortable"]),
            "n_state_2": int(state_counts["very uncomfortable"]),
        }
        row.update(metrics.to_dict())
        rows_summary.append(row)

    summary = pd.DataFrame(rows_summary).sort_values("n_transitions", ascending=False)
    return results, summary

In [ ]:
results_sunpair_bin, summary_sunpair_bin = analyze_sunpair_binary_metrics(
    df=df_analysis,
    state_col=STATE_COL,
    sun_col=SUN_COL,
    state_order=state_order,
    max_stop=MAX_STOP
)

summary_sunpair_bin

# ara fem boostrap i permutation per veure validesa de resultats

In [ ]:
def bootstrap_sunpair_binary_metrics(
    df_analysis,
    state_col="comfort3_option1",
    sun_col="sun",
    state_order=state_order,
    max_stop=5,
    n_boot=1000,
    seed=123
):
    rng = np.random.default_rng(seed)

    df = df_analysis.copy()
    df["traj_id"] = df["ID"].astype(str) + "||" + df["walk_id"].astype(str)
    traj_ids = df["traj_id"].drop_duplicates().tolist()

    rows = []

    for b in range(n_boot):
        sampled_ids = rng.choice(traj_ids, size=len(traj_ids), replace=True)

        parts = []
        for k, tid in enumerate(sampled_ids):
            sub = df[df["traj_id"] == tid].copy()
            new_id = f"{tid}__boot{k}"
            sub["ID"] = new_id
            sub["walk_id"] = new_id
            parts.append(sub)

        df_boot = pd.concat(parts, ignore_index=True)

        _, summary_boot = analyze_sunpair_binary_metrics(
            df=df_boot,
            state_col=state_col,
            sun_col=sun_col,
            state_order=state_order,
            max_stop=max_stop
        )

        summary_boot = summary_boot.copy()
        summary_boot["boot"] = b
        rows.append(summary_boot)

    boot_df = pd.concat(rows, ignore_index=True)
    return boot_df

In [ ]:
boot_sunpair_bin = bootstrap_sunpair_binary_metrics(
    df_analysis=df_analysis,
    state_col=STATE_COL,
    sun_col=SUN_COL,
    state_order=state_order,
    max_stop=MAX_STOP,
    n_boot=1000,
    seed=123
)

In [ ]:
def summarize_bootstrap_metrics(boot_df):
    metric_cols = [
        "onset_of_discomfort",
        "escalation_to_very_uncomfortable",
        "trap_persistence_very_uncomfortable",
        "recovery_from_uncomfortable",
        "escape_from_very_uncomfortable",
    ]

    rows = []

    for pair_value, sub in boot_df.groupby("sun_pair_bin"):
        for metric in metric_cols:
            vals = sub[metric].dropna().to_numpy()

            rows.append({
                "sun_pair_bin": pair_value,
                "metric": metric,
                "q025": float(np.quantile(vals, 0.025)),
                "median": float(np.quantile(vals, 0.5)),
                "q975": float(np.quantile(vals, 0.975)),
            })

    return pd.DataFrame(rows)

In [ ]:
boot_sunpair_bin_summary = summarize_bootstrap_metrics(boot_sunpair_bin)
boot_sunpair_bin_summary

In [ ]:
def observed_metrics_long(summary_df):
    metric_cols = [
        "onset_of_discomfort",
        "escalation_to_very_uncomfortable",
        "trap_persistence_very_uncomfortable",
        "recovery_from_uncomfortable",
        "escape_from_very_uncomfortable",
    ]

    rows = []
    for _, row in summary_df.iterrows():
        for metric in metric_cols:
            rows.append({
                "sun_pair_bin": row["sun_pair_bin"],
                "metric": metric,
                "observed": row[metric],
                "n_transitions": row["n_transitions"],
                "n_state_0": row["n_state_0"],
                "n_state_1": row["n_state_1"],
                "n_state_2": row["n_state_2"],
            })
    return pd.DataFrame(rows)

In [ ]:
obs_sunpair_long = observed_metrics_long(summary_sunpair_bin)

metrics_with_ci = obs_sunpair_long.merge(
    boot_sunpair_bin_summary,
    on=["sun_pair_bin", "metric"],
    how="left"
)

metrics_with_ci

# Traiem explanatory plots

In [ ]:
def plot_sunpair_metrics_with_ci(metrics_with_ci, pad_frac=0.08, min_pad=0.02):
    pair_order = [
        "not_sun_exposed -> not_sun_exposed",
        "not_sun_exposed -> sun_exposed",
        "sun_exposed -> not_sun_exposed",
        "sun_exposed -> sun_exposed",
    ]

    metric_order = [
        "onset_of_discomfort",
        "escalation_to_very_uncomfortable",
        "trap_persistence_very_uncomfortable",
        "recovery_from_uncomfortable",
        "escape_from_very_uncomfortable",
    ]

    metric_titles = {
        "onset_of_discomfort": "Onset of discomfort",
        "escalation_to_very_uncomfortable": "Escalation to very uncomfortable",
        "trap_persistence_very_uncomfortable": "Trap persistence (2→2)",
        "recovery_from_uncomfortable": "Recovery from uncomfortable ",
        "escape_from_very_uncomfortable": "Escape from very uncomfortable",
    }

    fig, axes = plt.subplots(len(metric_order), 1, figsize=(10, 16), sharex=True)

    for ax, metric in zip(axes, metric_order):
        sub = metrics_with_ci[metrics_with_ci["metric"] == metric].copy()
        sub["sun_pair_bin"] = pd.Categorical(
            sub["sun_pair_bin"],
            categories=pair_order,
            ordered=True
        )
        sub = sub.sort_values("sun_pair_bin").reset_index(drop=True)

        x = np.arange(len(sub))
        y = sub["observed"].to_numpy()
        q025 = sub["q025"].to_numpy()
        q975 = sub["q975"].to_numpy()

        yerr_low = y - q025
        yerr_high = q975 - y

        ax.errorbar(
            x, y,
            yerr=[yerr_low, yerr_high],
            fmt="o",
            capsize=5
        )

        # y-limits automàtics a partir dels intervals
        ymin_data = np.nanmin(q025)
        ymax_data = np.nanmax(q975)
        yrange = ymax_data - ymin_data

        pad = max(min_pad, pad_frac * yrange if yrange > 0 else min_pad)
        ymin = ymin_data - pad
        ymax = ymax_data + pad

        # opcional: retallar a [0,1] perquè són probabilitats
        ymin = max(0, ymin)
        ymax = min(1, ymax)

        # si per algun motiu queda massa estret
        if ymax <= ymin:
            ymin, ymax = max(0, ymin_data - 0.05), min(1, ymax_data + 0.05)

        ax.set_ylim(ymin-0.05, ymax+0.05)

        # anotacions
        text_offset = max(0.01, 0.04 * (ymax - ymin))
        for i, row in sub.iterrows():
            ax.text(
                i,
                row["observed"] + text_offset,
                f'{row["observed"]:.2f}',
                ha="center",
                va="bottom",
                fontsize=9
            )

        ax.set_ylabel("probability")
        ax.set_title(metric_titles[metric])

    axes[-1].set_xticks(np.arange(len(pair_order)))
    axes[-1].set_xticklabels(pair_order, rotation=35, ha="right")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_sunpair_metrics_with_ci(metrics_with_ci)

# Anem a mirar aquestes variables en diferentes condicions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

STATE_COL = "comfort3_option1"
MAX_STOP = 7

state_order = ["comfortable-neutral", "uncomfortable", "very uncomfortable"]

# toca netejar una mica les columnes que volem utilitzar

In [ ]:
df_grouped = df_analysis.copy()

def clean_gender(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    if x == "Man":
        return "Man"
    elif x == "Woman":
        return "Woman"
    else:
        return np.nan  # exclude small/ambiguous categories for stratified analysis


df_grouped["gender_clean"] = df_grouped["gender"].map(clean_gender)
df_grouped = df_grouped.dropna(subset=["gender_clean"]).copy()

def clean_age_group(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    if x in ["Less than 10", "10-12"]:
        return "<=12"

    elif x == "13-15":
        return "13-15"

    elif x in ["16-18", "19-24", "25-34"]:
        return "16-34"

    elif x in ["35-44", "45-54", "55-64", "65-74", "75-84", "85+"]:
        return "35+"

    else:
        return np.nan


df_grouped["age_group_clean"] = df_grouped["age"].map(clean_age_group)

def clean_city(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    city_map = {
        "L'Hospitalet de Llobreat": "L'Hospitalet de Llobregat",
    }

    return city_map.get(x, x)


df_grouped["city_clean"] = df_grouped["city"].map(clean_city)


def clean_spent_time(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    if x in ["Less than 30 minutes", "30 minutes – 1 hour", "30 minutes - 1 hour"]:
        return "<1h"

    elif x in ["1 hour – 2 hours", "1 hour - 2 hours"]:
        return "1-2h"

    elif x in ["2 hours – 4 hours", "2 hours - 4 hours", "More than 4 hours"]:
        return ">2h"

    else:
        return np.nan


df_grouped["spent_time_clean"] = df_grouped["spent_time"].map(clean_spent_time)

# construim l'analisi de transicions

In [ ]:
def build_transition_df_for_group_analysis(
    df,
    state_col="comfort3_option1",
    group_cols=None,
    max_stop=5
):
    if group_cols is None:
        group_cols = []

    df = df.copy()
    df = df.sort_values(["ID", "walk_id", "stop_idx", "row_id"]).reset_index(drop=True)

    g = df.groupby(["ID", "walk_id"], sort=False)

    df["state"] = df[state_col]
    df["next_state"] = g["state"].shift(-1)
    df["next_stop_idx"] = g["stop_idx"].shift(-1)

    transitions = df[
        (df["next_stop_idx"] - df["stop_idx"] == 1) &
        (df["stop_idx"] <= max_stop) &
        (df["next_stop_idx"] <= max_stop)
    ].copy()

    transitions = transitions.dropna(subset=["state", "next_state"]).copy()

    # Keep only group columns that exist
    existing_group_cols = [c for c in group_cols if c in transitions.columns]

    return transitions, existing_group_cols

In [ ]:
group_cols = [
    "gender_clean",
    "age_group_clean",
    "spent_time_clean",
    "city_clean",
    "participants_clothing",
]

transitions_group, existing_group_cols = build_transition_df_for_group_analysis(
    df_grouped,
    state_col=STATE_COL,
    group_cols=group_cols,
    max_stop=MAX_STOP
)

existing_group_cols

In [ ]:
group_cols_to_analyze = [
    c for c in [
        "gender_clean",
        "age_group_clean",
        "spent_time_clean",
        "city_clean",
        "participants_clothing",
    ]
    if c in transitions_group.columns
]

group_cols_to_analyze

In [ ]:
def build_event_df(transitions, event_name):
    df = transitions.copy()

    if event_name == "onset_of_discomfort":
        sub = df[df["state"] == "comfortable-neutral"].copy()
        sub["event"] = sub["next_state"].isin([
            "uncomfortable",
            "very uncomfortable"
        ]).astype(int)

    elif event_name == "escalation_to_very_uncomfortable":
        sub = df[df["state"] == "uncomfortable"].copy()
        sub["event"] = (sub["next_state"] == "very uncomfortable").astype(int)

    elif event_name == "trap_persistence_very_uncomfortable":
        sub = df[df["state"] == "very uncomfortable"].copy()
        sub["event"] = (sub["next_state"] == "very uncomfortable").astype(int)

    elif event_name == "recovery_from_uncomfortable":
        sub = df[df["state"] == "uncomfortable"].copy()
        sub["event"] = (sub["next_state"] == "comfortable-neutral").astype(int)

    elif event_name == "escape_from_very_uncomfortable":
        sub = df[df["state"] == "very uncomfortable"].copy()
        sub["event"] = sub["next_state"].isin([
            "comfortable-neutral",
            "uncomfortable"
        ]).astype(int)

    elif event_name == "discomfort_basin_persistence":
        sub = df[df["state"].isin([
            "uncomfortable",
            "very uncomfortable"
        ])].copy()
        sub["event"] = sub["next_state"].isin([
            "uncomfortable",
            "very uncomfortable"
        ]).astype(int)

    else:
        raise ValueError(f"Unknown event_name: {event_name}")

    return sub

In [ ]:
def event_rate_by_group(
    transitions,
    event_name,
    group_col,
    min_n=10
):
    event_df = build_event_df(transitions, event_name)

    rows = []

    for level, sub in event_df.groupby(group_col, dropna=False):
        n = len(sub)
        k = int(sub["event"].sum())

        if n == 0:
            continue

        rate = k / n
        ci_low, ci_high = wilson_ci(k, n)

        rows.append({
            "event_name": event_name,
            "group_col": group_col,
            "level": level,
            "n": n,
            "events": k,
            "rate": rate,
            "ci_low": ci_low,
            "ci_high": ci_high,
            "low_support": n < min_n,
        })

    out = pd.DataFrame(rows)

    if len(out) == 0:
        return out

    return out.sort_values(
        ["event_name", "rate", "n"],
        ascending=[True, False, False]
    )

In [ ]:
event_names = [
    "onset_of_discomfort",
    "escalation_to_very_uncomfortable",
    "discomfort_basin_persistence",
    "trap_persistence_very_uncomfortable",
    "recovery_from_uncomfortable",
    "escape_from_very_uncomfortable",
]

all_group_event_results = []

for event_name in event_names:
    for group_col in group_cols_to_analyze:
        tmp = event_rate_by_group(
            transitions_group,
            event_name=event_name,
            group_col=group_col,
            min_n=10
        )
        all_group_event_results.append(tmp)

all_group_event_results = pd.concat(all_group_event_results, ignore_index=True)

all_group_event_results.head(60)
all_group_event_results.to_csv("results_onset/group_event_rates.csv", index=False)

# i ara plotejem

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot_event_by_group_vertical(
    results_df,
    event_name,
    group_col,
    min_n=10,
    title=None,
    ylim_from_ci=True,
    pad_frac=0.10,
    min_pad=0.03,
):
    sub = results_df[
        (results_df["event_name"] == event_name) &
        (results_df["group_col"] == group_col) &
        (results_df["n"] >= min_n)
    ].copy()

    if sub.empty:
        print(f"No data for {event_name} by {group_col}")
        return

    custom_orders = {
        "gender_clean": ["Woman", "Man"],
        "age_group_clean": ["<=12", "13-15", "16-34", "35+"],
        "spent_time_clean": ["<1h", "1-2h", ">2h"],
    }

    group_titles = {
        "gender_clean": "Gender",
        "age_group_clean": "Age group",
        "spent_time_clean": "Time spent outdoors",
        "city_clean": "City",
    }

    event_titles = {
        "onset_of_discomfort": "Onset of discomfort",
        "escalation_to_very_uncomfortable": "Escalation to very uncomfortable",
        "discomfort_basin_persistence": "Discomfort-basin persistence",
        "trap_persistence_very_uncomfortable": "Very uncomfortable trap persistence",
        "recovery_from_uncomfortable": "Recovery from uncomfortable",
        "escape_from_very_uncomfortable": "Escape from very uncomfortable",
    }

    if group_col in custom_orders:
        sub["level"] = pd.Categorical(
            sub["level"],
            categories=custom_orders[group_col],
            ordered=True
        )
        sub = sub.sort_values("level")
    else:
        sub = sub.sort_values("rate", ascending=False)

    x = np.arange(len(sub))
    y = sub["rate"].to_numpy()

    yerr_low = y - sub["ci_low"].to_numpy()
    yerr_high = sub["ci_high"].to_numpy() - y

    labels = [
        f"{level}\n(n={int(n)})"
        for level, n in zip(sub["level"], sub["n"])
    ]

    fig, ax = plt.subplots(figsize=(max(7, 1.2 * len(sub)), 4.5))

    ax.errorbar(
        x,
        y,
        yerr=[yerr_low, yerr_high],
        fmt="o",
        capsize=5
    )

    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=35, ha="right")
    ax.set_ylabel("event probability")

    if title is None:
        title = f"{event_titles.get(event_name, event_name)} by {group_titles.get(group_col, group_col)}"

    ax.set_title(title)
    ax.grid(axis="y", alpha=0.25)

    if ylim_from_ci:
        ymin_data = sub["ci_low"].min()
        ymax_data = sub["ci_high"].max()
        yrange = ymax_data - ymin_data

        pad = max(min_pad, pad_frac * yrange if yrange > 0 else min_pad)

        ymin = max(0, ymin_data - pad)
        ymax = min(1, ymax_data + pad)

        if ymax <= ymin:
            ymin = max(0, ymin_data - 0.05)
            ymax = min(1, ymax_data + 0.05)

        ax.set_ylim(ymin, ymax)
    else:
        ax.set_ylim(0, 1)

    for i, row in enumerate(sub.itertuples()):
        ax.text(
            i,
            row.rate + 0.015,
            f"{row.rate:.2f}",
            ha="center",
            va="bottom",
            fontsize=9
        )

    plt.tight_layout()
    plt.show()

In [ ]:
plot_event_by_group_vertical(
    all_group_event_results,
    event_name="onset_of_discomfort",
    group_col="age_group_clean",
    min_n=10,
    title="Onset of discomfort by age group"
)
plot_event_by_group_vertical(
    all_group_event_results,
    event_name="onset_of_discomfort",
    group_col="gender_clean",
    min_n=10,
    title="Onset of discomfort by gender"
)
plot_event_by_group_vertical(
    all_group_event_results,
    event_name="onset_of_discomfort",
    group_col="spent_time_clean",
    min_n=10,
    title="Onset of discomfort by spent time"
)

plot_event_by_group_vertical(
    all_group_event_results,
    event_name="onset_of_discomfort",
    group_col="city_clean",
    min_n=10,
    title="Onset of discomfort by city"
)

In [ ]:
plot_event_by_group_vertical(
    all_group_event_results,
    event_name="escalation_to_very_uncomfortable",
    group_col="gender_clean",
    min_n=10,
    title="Escalation to very uncomfortable by age group")

plot_event_by_group_vertical(
    all_group_event_results,
    event_name="escalation_to_very_uncomfortable",
    group_col="age_group_clean",
    min_n=10,
    title="Escalation to very uncomfortable by age group")

plot_event_by_group_vertical(
    all_group_event_results,
    event_name="escalation_to_very_uncomfortable",
    group_col="spent_time_clean",
    min_n=10,
    title="Escalation to very uncomfortable by spent time")

plot_event_by_group_vertical(
    all_group_event_results,
    event_name="escalation_to_very_uncomfortable",
    group_col="city_clean",
    min_n=10,
    title="Escalation to very uncomfortable by city")

In [ ]:
plot_event_by_group_vertical(
    all_group_event_results,
    event_name="trap_persistence_very_uncomfortable",
    group_col="gender_clean",
    min_n=10,
    title="Trap persistence (2→2) by gender")

plot_event_by_group_vertical(
    all_group_event_results,
    event_name="trap_persistence_very_uncomfortable",
    group_col="age_group_clean",
    min_n=10,
    title="Trap persistence (2→2) by age group")

plot_event_by_group_vertical(
    all_group_event_results,
    event_name="trap_persistence_very_uncomfortable",
    group_col="spent_time_clean",
    min_n=10,
    title="Trap persistence (2→2) by spent time")

plot_event_by_group_vertical(
    all_group_event_results,
    event_name="trap_persistence_very_uncomfortable",
    group_col="city_clean",
    min_n=10,
    title="Trap persistence (2→2) by city")

In [ ]:
plot_event_by_group_vertical(
    all_group_event_results,
    event_name="recovery_from_uncomfortable",
    group_col="gender_clean",
    min_n=10,
    title="Recovery from uncomfortable by gender")

plot_event_by_group_vertical(
    all_group_event_results,
    event_name="recovery_from_uncomfortable",
    group_col="age_group_clean",
    min_n=10,
    title="Recovery from uncomfortable by age group")

plot_event_by_group_vertical(
    all_group_event_results,
    event_name="recovery_from_uncomfortable",
    group_col="spent_time_clean",
    min_n=10,
    title="Recovery from uncomfortable by spent time")

plot_event_by_group_vertical(
    all_group_event_results,
    event_name="recovery_from_uncomfortable",
    group_col="city_clean",
    min_n=10,
    title="Recovery from uncomfortable by city")

# i si maximitzem aquest onset?

In [ ]:
import numpy as np
import pandas as pd

STATE_COL = "comfort3_option1"
MAX_STOP = 5

state_order = ["comfortable-neutral", "uncomfortable", "very uncomfortable"]

continuous_vars = [
    "<T-T_fixed+<T>>",
    "<HDX-HDX_fixed+<HDX>>",
    "SVF_1.5m",
    "NDVI_1.5m",
    "IVAC",
    "distance_fountain",
    "distance_drinking_fountain",
    "distance_green_zone",
]

categorical_vars = [
    "sun",
    "wind",
    "climate_shelter",
    "surface_type",
    "space_category",
]

In [ ]:
def build_onset_transition_df(
    df,
    state_col=STATE_COL,
    continuous_vars=continuous_vars,
    categorical_vars=categorical_vars,
    max_stop=5
):
    df = df.copy()
    df = df.sort_values(["ID", "walk_id", "stop_idx", "row_id"]).reset_index(drop=True)

    g = df.groupby(["ID", "walk_id"], sort=False)

    df["state"] = df[state_col]
    df["next_state"] = g["state"].shift(-1)
    df["next_stop_idx"] = g["stop_idx"].shift(-1)

    # continuous variables at t, t+1, mean and delta
    for col in continuous_vars:
        if col in df.columns:
            df[f"{col}_t"] = df[col]
            df[f"{col}_t1"] = g[col].shift(-1)
            df[f"{col}_mean"] = 0.5 * (df[f"{col}_t"] + df[f"{col}_t1"])
            df[f"{col}_delta"] = df[f"{col}_t1"] - df[f"{col}_t"]

    # categorical variables at t, t+1 and pair
    for col in categorical_vars:
        if col in df.columns:
            df[f"{col}_t"] = df[col]
            df[f"{col}_t1"] = g[col].shift(-1)
            df[f"{col}_pair"] = df[f"{col}_t"].astype(str) + " -> " + df[f"{col}_t1"].astype(str)

    # special binary sun-pair
    if "sun" in df.columns:
        df["sunbin_t"] = df["sun"].map(recode_sun_binary)
        df["sunbin_t1"] = g["sun"].shift(-1).map(recode_sun_binary)
        df["sun_pair_bin"] = df["sunbin_t"].astype(str) + " -> " + df["sunbin_t1"].astype(str)

    transitions = df[
        (df["next_stop_idx"] - df["stop_idx"] == 1) &
        (df["stop_idx"] <= max_stop) &
        (df["next_stop_idx"] <= max_stop)
    ].copy()

    transitions = transitions.dropna(subset=["state", "next_state"]).copy()

    onset_df = transitions[transitions["state"] == "comfortable-neutral"].copy()

    onset_df["onset_discomfort"] = onset_df["next_state"].isin([
        "uncomfortable",
        "very uncomfortable"
    ]).astype(int)

    return transitions, onset_df

In [ ]:
transitions_features, onset_df = build_onset_transition_df(
    df_analysis,
    state_col=STATE_COL,
    continuous_vars=continuous_vars,
    categorical_vars=categorical_vars,
    max_stop=MAX_STOP
)

onset_df.shape
onset_df["onset_discomfort"].mean()

# funció de Wilson interval per a veure que m'augmenta l'onset

In [ ]:
def wilson_ci(k, n, z=1.96):
    if n == 0:
        return np.nan, np.nan

    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2*n)) / denom
    half = z * np.sqrt((p*(1-p)/n) + (z**2/(4*n**2))) / denom

    return center - half, center + half

In [ ]:
def recode_wind_binary(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()

    if x == "It's not windy":
        return "no_wind"
    else:
        return "wind_present"


def recode_wind_useful(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()

    if x in ["It's not windy", "A very light wind"]:
        return "weak_or_no_wind"
    else:
        return "useful_wind"

In [ ]:
def onset_rate_by_category(df, group_col, target_col="onset_discomfort", min_n=20):
    rows = []

    for val, sub in df.groupby(group_col, dropna=False):
        n = len(sub)
        k = int(sub[target_col].sum())
        rate = k / n if n > 0 else np.nan
        lo, hi = wilson_ci(k, n)

        rows.append({
            "variable": group_col,
            "level": val,
            "n": n,
            "events": k,
            "onset_rate": rate,
            "ci_low": lo,
            "ci_high": hi,
            "low_support": n < min_n,
        })

    return pd.DataFrame(rows).sort_values("onset_rate", ascending=False)

In [ ]:
cat_candidates = [
    "sun_pair_bin",
    "sun_pair",
    "wind_pair",
    "climate_shelter_pair",
    "surface_type_pair",
    "space_category_pair",
]

uni_cat_results = []

for col in cat_candidates:
    if col in onset_df.columns:
        tmp = onset_rate_by_category(onset_df, col, min_n=20)
        uni_cat_results.append(tmp)

uni_cat_results = pd.concat(uni_cat_results, ignore_index=True)
uni_cat_results.sort_values(["onset_rate", "n"], ascending=[False, False]).head(50)
uni_cat_results.to_csv("results_onset/onset_rate_by_category.csv", index=False)

In [ ]:
def add_quantile_bin(df, col, q=3, suffix="_bin"):
    df = df.copy()
    new_col = col + suffix
    df[new_col] = pd.qcut(
        df[col],
        q=q,
        labels=[f"Q{i+1}" for i in range(q)],
        duplicates="drop"
    )
    return df, new_col


In [ ]:
cont_candidates = []

for col in continuous_vars:
    for suffix in ["_t", "_t1", "_mean", "_delta"]:
        c = col + suffix
        if c in onset_df.columns:
            cont_candidates.append(c)

onset_binned = onset_df.copy()
bin_cols = []

for col in cont_candidates:
    if onset_binned[col].notna().sum() >= 50:
        onset_binned, bin_col = add_quantile_bin(onset_binned, col, q=3)
        bin_cols.append(bin_col)

uni_cont_results = []

for col in bin_cols:
    tmp = onset_rate_by_category(onset_binned, col, min_n=20)
    uni_cont_results.append(tmp)

uni_cont_results = pd.concat(uni_cont_results, ignore_index=True)
uni_cont_results.sort_values(["onset_rate", "n"], ascending=[False, False]).head(40)
uni_cont_results.to_csv("results_onset/onset_rate_by_binned_continuous.csv", index=False)

# combinacions

In [ ]:
from itertools import combinations

def subgroup_onset_search(
    df,
    feature_cols,
    target_col="onset_discomfort",
    max_depth=3,
    min_n=30
):
    rows = []
    global_rate = df[target_col].mean()

    for depth in range(1, max_depth + 1):
        for cols in combinations(feature_cols, depth):
            grouped = df.groupby(list(cols), dropna=False)

            for keys, sub in grouped:
                if not isinstance(keys, tuple):
                    keys = (keys,)

                n = len(sub)
                if n < min_n:
                    continue

                k = int(sub[target_col].sum())
                rate = k / n
                lo, hi = wilson_ci(k, n)

                rows.append({
                    "features": " + ".join(cols),
                    "levels": " | ".join([f"{c}={v}" for c, v in zip(cols, keys)]),
                    "depth": depth,
                    "n": n,
                    "events": k,
                    "onset_rate": rate,
                    "lift_vs_global": rate / global_rate if global_rate > 0 else np.nan,
                    "ci_low": lo,
                    "ci_high": hi,
                })

    out = pd.DataFrame(rows)

    if len(out) == 0:
        return out

    # ordenar pel límit inferior de l'interval, no només pel valor observat
    out = out.sort_values(["ci_low", "onset_rate", "n"], ascending=[False, False, False])
    return out

In [ ]:
feature_cols = []

preferred_features = [
    "sun_pair_bin",
    "<HDX-HDX_fixed+<HDX>>_mean_bin",
    "<HDX-HDX_fixed+<HDX>>_delta_bin",
    "<T-T_fixed+<T>>_mean_bin",
    "<T-T_fixed+<T>>_delta_bin",
    "wind_pair",
    "SVF_1.5m_mean_bin",
    "NDVI_1.5m_mean_bin",
    "climate_shelter_pair",
    "surface_type_pair",
    "space_category_pair",
]

for col in preferred_features:
    if col in onset_binned.columns:
        feature_cols.append(col)

feature_cols

In [ ]:
subgroup_results = subgroup_onset_search(
    onset_binned,
    feature_cols=feature_cols,
    target_col="onset_discomfort",
    max_depth=3,
    min_n=30
)

subgroup_results.head(30)

In [ ]:
subgroup_results.to_csv("results_onset/subgroup_onset_results.csv", index=False)

# màgia

In [ ]:
import numpy as np
import pandas as pd

STATE_COL = "comfort3_option1"
MAX_STOP = 5

state_order = ["comfortable-neutral", "uncomfortable", "very uncomfortable"]

continuous_vars = [
    "<T-T_fixed+<T>>",
    "<HDX-HDX_fixed+<HDX>>",
    "SVF_1.5m",
    "NDVI_1.5m",
    "IVAC",
    "distance_fountain",
    "distance_drinking_fountain",
    "distance_green_zone",
]

categorical_vars = [
    "sun",
    "wind",
    "climate_shelter",
    "surface_type",
    "space_category",
]

In [ ]:
def recode_wind_binary(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    if x == "It's not windy":
        return "no_wind"
    else:
        return "wind_present"

In [ ]:
def build_transition_feature_df(
    df,
    state_col=STATE_COL,
    continuous_vars=continuous_vars,
    categorical_vars=categorical_vars,
    max_stop=5,
):
    df = df.copy()
    df = df.sort_values(["ID", "walk_id", "stop_idx", "row_id"]).reset_index(drop=True)

    g = df.groupby(["ID", "walk_id"], sort=False)

    # State at t and t+1
    df["state"] = df[state_col]
    df["next_state"] = g["state"].shift(-1)
    df["next_stop_idx"] = g["stop_idx"].shift(-1)

    # Continuous variables: t, t+1, mean, delta
    for col in continuous_vars:
        if col in df.columns:
            df[f"{col}_t"] = df[col]
            df[f"{col}_t1"] = g[col].shift(-1)
            df[f"{col}_mean"] = 0.5 * (df[f"{col}_t"] + df[f"{col}_t1"])
            df[f"{col}_delta"] = df[f"{col}_t1"] - df[f"{col}_t"]

    # Raw categorical pairs
    for col in categorical_vars:
        if col in df.columns:
            df[f"{col}_t"] = df[col]
            df[f"{col}_t1"] = g[col].shift(-1)
            df[f"{col}_pair"] = (
                df[f"{col}_t"].astype(str) + " -> " + df[f"{col}_t1"].astype(str)
            )

    # Binary sun pair
    if "sun" in df.columns:
        df["sunbin_t"] = df["sun"].map(recode_sun_binary)
        df["sunbin_t1"] = g["sun"].shift(-1).map(recode_sun_binary)
        df["sun_pair_bin"] = (
            df["sunbin_t"].astype(str) + " -> " + df["sunbin_t1"].astype(str)
        )

    # Binary wind pair: no wind vs some wind
    if "wind" in df.columns:
        df["windbin_t"] = df["wind"].map(recode_wind_binary)
        df["windbin_t1"] = g["wind"].shift(-1).map(recode_wind_binary)
        df["wind_pair_bin"] = (
            df["windbin_t"].astype(str) + " -> " + df["windbin_t1"].astype(str)
        )

        # Optional useful-wind pair
        df["winduse_t"] = df["wind"].map(recode_wind_useful)
        df["winduse_t1"] = g["wind"].shift(-1).map(recode_wind_useful)
        df["wind_pair_useful"] = (
            df["winduse_t"].astype(str) + " -> " + df["winduse_t1"].astype(str)
        )

    # Keep valid consecutive transitions
    transitions = df[
        (df["next_stop_idx"] - df["stop_idx"] == 1) &
        (df["stop_idx"] <= max_stop) &
        (df["next_stop_idx"] <= max_stop)
    ].copy()

    transitions = transitions.dropna(subset=["state", "next_state"]).copy()

    return transitions

In [ ]:
transitions_features = build_transition_feature_df(
    df_analysis,
    state_col=STATE_COL,
    continuous_vars=continuous_vars,
    categorical_vars=categorical_vars,
    max_stop=MAX_STOP,
)

transitions_features.shape

In [ ]:
def add_quantile_bins(df, cols, q=3):
    df = df.copy()
    bin_cols = []

    for col in cols:
        if col not in df.columns:
            continue

        if df[col].notna().sum() < 30:
            continue

        new_col = col + "_bin"

        try:
            df[new_col] = pd.qcut(
                df[col],
                q=q,
                labels=[f"Q{i+1}" for i in range(q)],
                duplicates="drop"
            )
            bin_cols.append(new_col)
        except ValueError:
            pass

    return df, bin_cols

In [ ]:
def build_event_df(transitions, event_name):
    df = transitions.copy()

    if event_name == "onset_of_discomfort":
        # Only transitions starting from comfortable-neutral
        sub = df[df["state"] == "comfortable-neutral"].copy()
        sub["event"] = sub["next_state"].isin([
            "uncomfortable",
            "very uncomfortable"
        ]).astype(int)

    elif event_name == "escalation_to_very_uncomfortable":
        # Only transitions starting from uncomfortable
        sub = df[df["state"] == "uncomfortable"].copy()
        sub["event"] = (sub["next_state"] == "very uncomfortable").astype(int)

    elif event_name == "trap_persistence_very_uncomfortable":
        # Only transitions starting from very uncomfortable
        sub = df[df["state"] == "very uncomfortable"].copy()
        sub["event"] = (sub["next_state"] == "very uncomfortable").astype(int)

    elif event_name == "recovery_from_uncomfortable":
        # Only transitions starting from uncomfortable
        sub = df[df["state"] == "uncomfortable"].copy()
        sub["event"] = (sub["next_state"] == "comfortable-neutral").astype(int)

    elif event_name == "escape_from_very_uncomfortable":
        # Only transitions starting from very uncomfortable
        sub = df[df["state"] == "very uncomfortable"].copy()
        sub["event"] = sub["next_state"].isin([
            "comfortable-neutral",
            "uncomfortable"
        ]).astype(int)

    else:
        raise ValueError(f"Unknown event_name: {event_name}")

    return sub

In [ ]:
onset_df = build_event_df(transitions_features, "onset_of_discomfort")
onset_df["event"].mean(), onset_df.shape

In [ ]:
def wilson_ci(k, n, z=1.96):
    if n == 0:
        return np.nan, np.nan

    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2*n)) / denom
    half = z * np.sqrt((p * (1 - p) / n) + (z**2 / (4 * n**2))) / denom

    return center - half, center + half

In [ ]:
def event_rate_by_category(event_df, group_col, target_col="event", min_n=20):
    rows = []

    for val, sub in event_df.groupby(group_col, dropna=False):
        n = len(sub)
        k = int(sub[target_col].sum())
        rate = k / n if n > 0 else np.nan
        lo, hi = wilson_ci(k, n)

        rows.append({
            "variable": group_col,
            "level": val,
            "n": n,
            "events": k,
            "event_rate": rate,
            "ci_low": lo,
            "ci_high": hi,
            "low_support": n < min_n,
        })

    return pd.DataFrame(rows).sort_values(
        ["event_rate", "n"],
        ascending=[False, False]
    )

In [ ]:
def analyze_event_univariate(
    transitions_features,
    event_name,
    group_cols,
    min_n=20
):
    event_df = build_event_df(transitions_features, event_name)

    results = []

    for col in group_cols:
        if col not in event_df.columns:
            continue

        tmp = event_rate_by_category(event_df, col, min_n=min_n)
        tmp["event_name"] = event_name
        results.append(tmp)

    if len(results) == 0:
        return pd.DataFrame()

    out = pd.concat(results, ignore_index=True)
    return out

In [ ]:
group_cols = [
    # sun
    "sun_pair_bin",
    "sun_pair",

    # wind
    "wind_pair_bin",
    "wind_pair_useful",
    "wind_pair",

    # other categorical transitions
    "climate_shelter_pair",
    "surface_type_pair",
    "space_category_pair",

    # continuous bins
    "<T-T_fixed+<T>>_t_bin",
    "<T-T_fixed+<T>>_t1_bin",
    "<T-T_fixed+<T>>_mean_bin",
    "<T-T_fixed+<T>>_delta_bin",

    "<HDX-HDX_fixed+<HDX>>_t_bin",
    "<HDX-HDX_fixed+<HDX>>_t1_bin",
    "<HDX-HDX_fixed+<HDX>>_mean_bin",
    "<HDX-HDX_fixed+<HDX>>_delta_bin",

    "SVF_1.5m_mean_bin",
    "SVF_1.5m_delta_bin",

    "NDVI_1.5m_mean_bin",
    "NDVI_1.5m_delta_bin",
]

In [ ]:
onset_univariate = analyze_event_univariate(
    transitions_features,
    event_name="onset_of_discomfort",
    group_cols=group_cols,
    min_n=20
)

onset_univariate.sort_values(["ci_low", "event_rate", "n"], ascending=[False, False, False]).head(30)
onset_univariate.to_csv("results_onset/2onset_univariate_analysis.csv", index=False)

In [ ]:
event_names = [
    "onset_of_discomfort",
    "escalation_to_very_uncomfortable",
    "trap_persistence_very_uncomfortable",
    "recovery_from_uncomfortable",
    "escape_from_very_uncomfortable",
]

all_event_univariate = []

for event_name in event_names:
    tmp = analyze_event_univariate(
        transitions_features,
        event_name=event_name,
        group_cols=group_cols,
        min_n=20
    )
    all_event_univariate.append(tmp)

all_event_univariate = pd.concat(all_event_univariate, ignore_index=True)
all_event_univariate.to_csv("results_onset/all_event_univariate_analysis.csv", index=False)

In [ ]:
trap_univariate = analyze_event_univariate(
    transitions_features,
    event_name="trap_persistence_very_uncomfortable",
    group_cols=group_cols,
    min_n=10
)

trap_univariate.sort_values(["ci_low", "event_rate", "n"], ascending=[False, False, False]).head(30)
trap_univariate.to_csv("results_onset/trap_persistence_univariate.csv", index=False)

In [ ]:
from itertools import combinations

def subgroup_event_search(
    event_df,
    feature_cols,
    target_col="event",
    max_depth=3,
    min_n=30
):
    rows = []
    global_rate = event_df[target_col].mean()

    for depth in range(1, max_depth + 1):
        for cols in combinations(feature_cols, depth):

            grouped = event_df.groupby(list(cols), dropna=False)

            for keys, sub in grouped:
                if not isinstance(keys, tuple):
                    keys = (keys,)

                n = len(sub)
                if n < min_n:
                    continue

                k = int(sub[target_col].sum())
                rate = k / n
                lo, hi = wilson_ci(k, n)

                rows.append({
                    "features": " + ".join(cols),
                    "levels": " | ".join([f"{c}={v}" for c, v in zip(cols, keys)]),
                    "depth": depth,
                    "n": n,
                    "events": k,
                    "event_rate": rate,
                    "lift_vs_global": rate / global_rate if global_rate > 0 else np.nan,
                    "ci_low": lo,
                    "ci_high": hi,
                })

    out = pd.DataFrame(rows)

    if len(out) == 0:
        return out

    return out.sort_values(
        ["ci_low", "event_rate", "n"],
        ascending=[False, False, False]
    )

In [ ]:
subgroup_features = [
    "sun_pair_bin",
    "wind_pair_bin",
    "wind_pair_useful",

    "<HDX-HDX_fixed+<HDX>>_mean_bin",
    "<HDX-HDX_fixed+<HDX>>_delta_bin",

    "<T-T_fixed+<T>>_mean_bin",
    "<T-T_fixed+<T>>_delta_bin",

    "SVF_1.5m_mean_bin",
    "NDVI_1.5m_mean_bin",
    "climate_shelter_pair",
    "surface_type_pair",
    "space_category_pair",
]

subgroup_features = [c for c in subgroup_features if c in transitions_features.columns]
subgroup_features

In [ ]:
onset_df = build_event_df(transitions_features, "onset_of_discomfort")

onset_subgroups = subgroup_event_search(
    onset_df,
    feature_cols=subgroup_features,
    max_depth=3,
    min_n=30
)

onset_subgroups.head(30)
onset_subgroups.to_csv("results_onset/2onset_subgroup_search.csv", index=False)

In [ ]:
escalation_df = build_event_df(transitions_features, "escalation_to_very_uncomfortable")

escalation_subgroups = subgroup_event_search(
    escalation_df,
    feature_cols=subgroup_features,
    max_depth=2,   # usually safer because n is smaller
    min_n=20
)

escalation_subgroups.head(30)
escalation_subgroups.to_csv("results_onset/escalation_subgroup_search.csv", index=False)

In [ ]:
trap_df = build_event_df(transitions_features, "trap_persistence_very_uncomfortable")

trap_subgroups = subgroup_event_search(
    trap_df,
    feature_cols=subgroup_features,
    max_depth=2,
    min_n=10
)

trap_subgroups.head(30)
trap_subgroups.to_csv("results_onset/trap_persistence_subgroup_search.csv", index=False)

# i alguns plots

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

all_event = pd.read_csv("results_onset/all_event_univariate_analysis.csv")

selected_variables = [
    "sun_pair_bin",
    "wind_pair_bin",
    "wind_pair_useful",
]

selected_events = [
    "onset_of_discomfort",
    "escalation_to_very_uncomfortable",
    "trap_persistence_very_uncomfortable",
    "recovery_from_uncomfortable",
    "escape_from_very_uncomfortable",
]

plot_df = all_event[
    all_event["variable"].isin(selected_variables) &
    all_event["event_name"].isin(selected_events) &
    (~all_event["low_support"])
].copy()

In [ ]:
def plot_event_rates_for_variable(all_event, variable, selected_events=None, min_n=20):
    df = all_event[all_event["variable"] == variable].copy()

    if selected_events is not None:
        df = df[df["event_name"].isin(selected_events)].copy()

    df = df[df["n"] >= min_n].copy()

    event_order = [
        "onset_of_discomfort",
        "escalation_to_very_uncomfortable",
        "trap_persistence_very_uncomfortable",
        "recovery_from_uncomfortable",
        "escape_from_very_uncomfortable",
    ]

    df["event_name"] = pd.Categorical(df["event_name"], categories=event_order, ordered=True)
    df = df.sort_values(["event_name", "event_rate"], ascending=[True, False])

    events = [e for e in event_order if e in df["event_name"].astype(str).unique()]

    for event in events:
        sub = df[df["event_name"].astype(str) == event].copy()
        if sub.empty:
            continue

        sub = sub.sort_values("event_rate", ascending=True)

        y = np.arange(len(sub))
        x = sub["event_rate"].to_numpy()
        xerr_low = x - sub["ci_low"].to_numpy()
        xerr_high = sub["ci_high"].to_numpy() - x

        fig, ax = plt.subplots(figsize=(8, max(3, 0.45 * len(sub))))

        ax.errorbar(
            x,
            y,
            xerr=[xerr_low, xerr_high],
            fmt="o",
            capsize=4
        )

        labels = [
            f"{lvl}\n(n={n})"
            for lvl, n in zip(sub["level"], sub["n"])
        ]

        ax.set_yticks(y)
        ax.set_yticklabels(labels)
        ax.set_xlim(0, min(1, max(sub["ci_high"].max() + 0.08, 0.35)))
        ax.set_xlabel("event probability")
        ax.set_title(f"{event} by {variable}")
        ax.axvline(sub["event_rate"].mean(), linestyle="--", linewidth=1)

        plt.tight_layout()
        plt.show()

In [ ]:
for var in ["wind_pair_bin", "wind_pair_useful", "sun_pair_bin"]:
    plot_event_rates_for_variable(
        all_event,
        variable=var,
        selected_events=selected_events,
        min_n=20
    )

# + plots per les nostres vars

In [ ]:
onset_subgroups = pd.read_csv("results_onset/2onset_subgroup_search.csv")
escalation_subgroups = pd.read_csv("results_onset/escalation_subgroup_search.csv")
trap_subgroups = pd.read_csv("results_onset/trap_persistence_subgroup_search.csv")

In [ ]:
def plot_top_subgroups(subgroup_df, title, top_n=10, min_n=20):
    df = subgroup_df[subgroup_df["n"] >= min_n].copy()

    # Sort by lower CI, not just observed event rate
    df = df.sort_values(
        ["ci_low", "event_rate", "n"],
        ascending=[False, False, False]
    ).head(top_n)

    # Make readable label
    df["label"] = df["levels"].str.replace(" | ", "\n", regex=False)

    df = df.sort_values("event_rate", ascending=True)

    y = np.arange(len(df))
    x = df["event_rate"].to_numpy()
    xerr_low = x - df["ci_low"].to_numpy()
    xerr_high = df["ci_high"].to_numpy() - x

    fig, ax = plt.subplots(figsize=(10, max(4, 0.75 * len(df))))

    ax.errorbar(
        x,
        y,
        xerr=[xerr_low, xerr_high],
        fmt="o",
        capsize=4
    )

    labels = [
        f"{lab}\n(n={n}, lift={lift:.2f})"
        for lab, n, lift in zip(df["label"], df["n"], df["lift_vs_global"])
    ]

    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.set_xlabel("event probability")
    ax.set_title(title)
    ax.set_xlim(0, min(1, max(df["ci_high"].max() + 0.08, 0.5)))

    plt.tight_layout()
    plt.show()

In [ ]:
plot_top_subgroups(
    onset_subgroups,
    title="Top robust onset-of-discomfort regimes",
    top_n=10,
    min_n=30
)

plot_top_subgroups(
    escalation_subgroups,
    title="Top escalation-to-very-uncomfortable regimes",
    top_n=10,
    min_n=20
)

plot_top_subgroups(
    trap_subgroups,
    title="Top trap-persistence regimes",
    top_n=8,
    min_n=10
)

# mmecàniques concretes

In [ ]:
mechanism_levels = {
    "wind_pair_bin": [
        "wind_present -> no_wind",
        "no_wind -> no_wind",
        "wind_present -> wind_present",
        "no_wind -> wind_present",
    ],
    "sun_pair_bin": [
        "not_sun_exposed -> sun_exposed",
        "sun_exposed -> sun_exposed",
        "sun_exposed -> not_sun_exposed",
        "not_sun_exposed -> not_sun_exposed",
    ],
}

In [ ]:
def plot_mechanism_summary(all_event, mechanism_levels, selected_events, min_n=10):
    rows = []

    for variable, levels in mechanism_levels.items():
        sub = all_event[
            (all_event["variable"] == variable) &
            (all_event["level"].isin(levels)) &
            (all_event["event_name"].isin(selected_events)) &
            (all_event["n"] >= min_n)
        ].copy()

        rows.append(sub)

    df = pd.concat(rows, ignore_index=True)

    for event in selected_events:
        sub = df[df["event_name"] == event].copy()
        if sub.empty:
            continue

        sub["condition"] = sub["variable"] + ": " + sub["level"]
        sub = sub.sort_values("event_rate", ascending=True)

        y = np.arange(len(sub))
        x = sub["event_rate"].to_numpy()
        xerr_low = x - sub["ci_low"].to_numpy()
        xerr_high = sub["ci_high"].to_numpy() - x

        fig, ax = plt.subplots(figsize=(9, max(4, 0.45 * len(sub))))

        ax.errorbar(
            x,
            y,
            xerr=[xerr_low, xerr_high],
            fmt="o",
            capsize=4
        )

        labels = [
            f"{cond}\n(n={n})"
            for cond, n in zip(sub["condition"], sub["n"])
        ]

        ax.set_yticks(y)
        ax.set_yticklabels(labels)
        ax.set_xlabel("event probability")
        ax.set_title(event)
        ax.set_xlim(0, min(1, max(sub["ci_high"].max() + 0.08, 0.5)))

        plt.tight_layout()
        plt.show()

In [ ]:
plot_mechanism_summary(
    all_event,
    mechanism_levels=mechanism_levels,
    selected_events=[
        "onset_of_discomfort",
        "escalation_to_very_uncomfortable",
        "trap_persistence_very_uncomfortable",
        "recovery_from_uncomfortable",
        "escape_from_very_uncomfortable",
    ],
    min_n=10
)

# altri

In [ ]:
def segment_onset_concordance(transitions_features, min_at_risk=5):
    df = transitions_features.copy()

    onset_df = df[df["state"] == "comfortable-neutral"].copy()
    onset_df["onset"] = onset_df["next_state"].isin([
        "uncomfortable",
        "very uncomfortable"
    ]).astype(int)

    group_cols = ["walk_id", "stop_idx", "next_stop_idx"]

    rows = []

    for keys, sub in onset_df.groupby(group_cols, dropna=False):
        walk_id, stop_idx, next_stop_idx = keys

        n = len(sub)
        k = int(sub["onset"].sum())

        if n < min_at_risk:
            continue

        rate = k / n
        ci_low, ci_high = wilson_ci(k, n)

        row = {
            "walk_id": walk_id,
            "stop_idx": stop_idx,
            "next_stop_idx": next_stop_idx,
            "n_at_risk": n,
            "n_onset": k,
            "onset_rate_segment": rate,
            "ci_low": ci_low,
            "ci_high": ci_high,
        }

        # modal categorical transition conditions within the segment
        cat_cols = [
            "sun_pair_bin",
            "sun_pair",
            "wind_pair_bin",
            "wind_pair_useful",
            "wind_pair",
            "climate_shelter_pair",
            "surface_type_pair",
            "space_category_pair",
        ]

        for col in cat_cols:
            if col in sub.columns:
                mode_val = sub[col].mode(dropna=True)
                row[col] = mode_val.iloc[0] if len(mode_val) else np.nan

        # mean continuous transition features
        cont_cols = [
            "<T-T_fixed+<T>>_mean",
            "<T-T_fixed+<T>>_delta",
            "<HDX-HDX_fixed+<HDX>>_mean",
            "<HDX-HDX_fixed+<HDX>>_delta",
            "SVF_1.5m_mean",
            "NDVI_1.5m_mean",
            "IVAC_mean",
        ]

        for col in cont_cols:
            if col in sub.columns:
                row[col] = sub[col].mean()

        # modal binned features if present
        bin_cols = [
            "<T-T_fixed+<T>>_mean_bin",
            "<T-T_fixed+<T>>_delta_bin",
            "<HDX-HDX_fixed+<HDX>>_mean_bin",
            "<HDX-HDX_fixed+<HDX>>_delta_bin",
            "SVF_1.5m_mean_bin",
            "NDVI_1.5m_mean_bin",
        ]

        for col in bin_cols:
            if col in sub.columns:
                mode_val = sub[col].mode(dropna=True)
                row[col] = mode_val.iloc[0] if len(mode_val) else np.nan

        rows.append(row)

    out = pd.DataFrame(rows)

    if out.empty:
        return out

    out = out.sort_values(
        ["ci_low", "onset_rate_segment", "n_at_risk"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    return out

In [ ]:
segment_onset = segment_onset_concordance(
    transitions_features,
    min_at_risk=5
)

segment_onset.head(20)

In [ ]:
def add_high_risk_condition_flags(segment_df):
    df = segment_df.copy()

    if "wind_pair_bin" in df.columns:
        df["flag_wind_loss"] = df["wind_pair_bin"].eq("wind_present -> no_wind")
        df["flag_no_wind_persistent"] = df["wind_pair_bin"].eq("no_wind -> no_wind")

    if "wind_pair_useful" in df.columns:
        df["flag_useful_to_weak_wind"] = df["wind_pair_useful"].eq("useful_wind -> weak_or_no_wind")
        df["flag_weak_wind_persistent"] = df["wind_pair_useful"].eq("weak_or_no_wind -> weak_or_no_wind")

    if "sun_pair_bin" in df.columns:
        df["flag_entering_sun"] = df["sun_pair_bin"].eq("not_sun_exposed -> sun_exposed")
        df["flag_persistent_sun"] = df["sun_pair_bin"].eq("sun_exposed -> sun_exposed")
        df["flag_leaving_sun"] = df["sun_pair_bin"].eq("sun_exposed -> not_sun_exposed")
        df["flag_persistent_no_sun"] = df["sun_pair_bin"].eq("not_sun_exposed -> not_sun_exposed")

    if "sun_pair" in df.columns:
        df["flag_entering_full_sun"] = df["sun_pair"].astype(str).str.endswith("-> In full sun")

    # Quantile-bin high thermal conditions, if present
    for col, flag_name in [
        ("<HDX-HDX_fixed+<HDX>>_mean_bin", "flag_high_HDX_mean"),
        ("<HDX-HDX_fixed+<HDX>>_delta_bin", "flag_high_HDX_delta"),
        ("<T-T_fixed+<T>>_mean_bin", "flag_high_T_mean"),
        ("<T-T_fixed+<T>>_delta_bin", "flag_high_T_delta"),
    ]:
        if col in df.columns:
            df[flag_name] = df[col].astype(str).eq("Q3")

    flag_cols = [c for c in df.columns if c.startswith("flag_")]
    df["n_high_risk_flags"] = df[flag_cols].sum(axis=1)

    return df

In [ ]:
segment_onset_flags = add_high_risk_condition_flags(segment_onset)

segment_onset_flags.head(20)

In [ ]:
cols_to_show = [
    "walk_id",
    "stop_idx",
    "next_stop_idx",
    "n_at_risk",
    "n_onset",
    "onset_rate_segment",
    "ci_low",
    "ci_high",
    "sun_pair_bin",
    "wind_pair_bin",
    "wind_pair_useful",
    "<HDX-HDX_fixed+<HDX>>_mean_bin",
    "<HDX-HDX_fixed+<HDX>>_delta_bin",
    "<T-T_fixed+<T>>_mean_bin",
    "<T-T_fixed+<T>>_delta_bin",
    "n_high_risk_flags",
]

cols_to_show = [c for c in cols_to_show if c in segment_onset_flags.columns]

segment_onset_flags[cols_to_show].head(20)

In [ ]:
top_k = 15
top_segments = segment_onset_flags.head(top_k).copy()

flag_cols = [c for c in top_segments.columns if c.startswith("flag_")]

flag_summary_top = (
    top_segments[flag_cols]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

flag_summary_top.columns = ["condition_flag", f"fraction_in_top_{top_k}_segments"]
flag_summary_top

In [ ]:
flag_summary_all = (
    segment_onset_flags[flag_cols]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

flag_summary_all.columns = ["condition_flag", "fraction_in_all_segments"]

flag_comparison = flag_summary_top.merge(
    flag_summary_all,
    on="condition_flag",
    how="left"
)

flag_comparison["enrichment_top_vs_all"] = (
    flag_comparison[f"fraction_in_top_{top_k}_segments"] /
    flag_comparison["fraction_in_all_segments"].replace(0, np.nan)
)

flag_comparison

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_top_onset_segments(segment_df, top_n=15):
    df = segment_df.sort_values(
        ["ci_low", "onset_rate_segment", "n_at_risk"],
        ascending=[False, False, False]
    ).head(top_n).copy()

    df = df.sort_values("onset_rate_segment", ascending=True)

    labels = []
    for _, row in df.iterrows():
        label = (
            f"walk {row['walk_id']} | {row['stop_idx']}→{row['next_stop_idx']}"
            f"\nn={int(row['n_at_risk'])}, onset={int(row['n_onset'])}"
        )

        if "sun_pair_bin" in row:
            label += f"\n{row['sun_pair_bin']}"
        if "wind_pair_bin" in row:
            label += f" | {row['wind_pair_bin']}"

        labels.append(label)

    y = np.arange(len(df))
    x = df["onset_rate_segment"].to_numpy()
    xerr_low = x - df["ci_low"].to_numpy()
    xerr_high = df["ci_high"].to_numpy() - x

    fig, ax = plt.subplots(figsize=(10, max(5, 0.8 * len(df))))

    ax.errorbar(
        x,
        y,
        xerr=[xerr_low, xerr_high],
        fmt="o",
        capsize=4
    )

    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.set_xlabel("segment onset rate")
    ax.set_title("Segments with strongest participant-level onset concordance")
    ax.set_xlim(0, 1)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_top_onset_segments(segment_onset_flags, top_n=15)